# Introduction to the PIC-SURE API
This is a tutorial notebook aimed to get the user quickly up and running with the PIC-SURE API. 

## PIC-SURE python API
### What is PIC-SURE?
As part of the *NHLBI BioData Catalyst® (BDC)* ecosystem, the Patient Information Commons: Standard Unification of Research Elements (PIC-SURE) platform has been integrating clinical and genomic datasets from multiple TOPMed and TOPMed-related studies funded by the National Heart, Lung, and Blood Institute (NHLBI). 

Original data exposed through the PIC-SURE API encompasses a large heterogeneity of data organization underneath. PIC-SURE hides this complexity and esposes the different study datasets in a single tabular format. By simplifying the process of data extraction, it allows investigators to focus on downstream analysis and to facilitate reproducible science. 

### More about PIC-SURE
The API is available in two different programming languages, python and R, enabling investigators to query the database the same way using either language.

PIC-SURE is a larger project from which the R and python PIC-SURE APIs are only a small part. Among other things, PIC-SURE also offers a graphical user interface that allows researchers to explore variables across multiple studies, filter participants that match criteria, and create cohorts from this interactive exploration.

The python API is actively developed by the Avillach Lab at Harvard Medical School. 

PIC-SURE API GitHub repositories:
* https://github.com/hms-dbmi/pic-sure-python-adapter-hpds

 ------- 

## Getting your user-specific security token

**Before running this notebook, please be sure to review the "Get your security token" documentation, which exists in the [`README.md` file](../README.md). It explains how to get a security token, which is mandatory to use the PIC-SURE API.**

To set up your token file, be sure to run the [`Workspace_setup.ipynb` file](./Workspace_setup.ipynb).

## Environment set-up

### Pre-requisites
* python 3.10.20 or later
* pip python package manager, already available in most systems with a python interpreter installed

### Install packages
The first step to using the PIC-SURE API is to install the `picsure` package from GitHub.

**Note that if you are using the dedicated PIC-SURE environment within the *BDC Powered by Seven Bridges* platform, the necessary packages have already been installed.**

In [1]:
import sys
import pandas as pd
# pip install matplotlib
import matplotlib.pyplot as plt
# BDC Powered by Terra users uncomment the following line to specify package install location
# sys.path.insert(0, r"/home/jupyter/.local/lib/python3.7/site-packages")

In [2]:
!{sys.executable} -m uv pip install --upgrade --force-reinstall git+https://github.com/hms-dbmi/pic-sure-python-adapter-hpds.git@ALS-11934-include-filter-variables

/Users/george/code_workspaces/adapters/pic-sure-python-adapter-hpds/.venv/bin/python3.10: No module named uv


In [3]:
import picsure

## Connecting to a PIC-SURE resource

The following is required to get access to the PIC-SURE API:
* a network URL
* a user-specific security token

The following code specifies the network URL as the *BDC Powered by PIC-SURE* URL and references the user-specific token saved as `token.txt`.

If you have not already retrieved your user-specific token, please refer to the "Get your security token" section of the `README.md` file and the `Workspace_setup.ipynb` file.

In [4]:
token_file = "token.txt"

with open(token_file, "r") as f:
    my_token = f.read()

session = picsure.connect(picsure.Platform.BDC_AUTHORIZED, my_token)

You're successfully connected to BDC Authorized as user george_colon@hms.harvard.edu!
Your token expires on 2026-06-25T21:56:48Z.


## Exploring available resources
You can inspect the resources available to your session using `session.getResourceID()`, which returns the resource UUIDs your token has access to.

In [5]:
session.getResourceID()

,uuid,name,description
0,ac004461-1b47-4832-80e2-22a4aecabe39,open-hpds-v3,
1,ca0ad4a9-130a-3a8a-ae00-e35b07f1108b,visualization,
2,02e23f52-f354-4e8b-992c-d37c8b9ba140,auth-hpds,
3,36363664-6231-6134-2d38-6538652d3131,dictionary,


The output above shows the resource UUIDs available to your session.

## Using the PIC-SURE variables dictionary
For the rest of this example notebook, we will use one of the publicly available datasets available on PIC-SURE. This dataset is the "Framingham Heart Study: Dataset for Teaching Purposes", which has a study accession of `tutorial-biolincc_framingham` in PIC-SURE. To find the variables related to this dataset, we will use `session.searchDictionary()` to search for a term of interest - in this case, `tutorial`.

In [6]:
search_term = "tutorial"

In [7]:
my_variables_df = session.searchDictionary(search_term)  # Search for "tutorial"

In [8]:
my_variables_df.shape[0]  # How many variables did the search return?

139

We can view these variables in a detailed DataFrame format; `session.searchDictionary()` returns a pandas DataFrame directly, so we call `.head(5)` on the result.

In [9]:
my_variables_df.head(5)  # View the first 5 rows

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\tutorial-biolincc_camp\AGEHOME\,AGEHOME,AGEHOME,,Categorical,tutorial-biolincc_camp,"[0, 1, 10, 100, 102, 103, 104, 105, 11, 110, 1...",NaN,NaN,True,None,biolincc_camp
1,\tutorial-biolincc_camp\AGE_RZ\,AGE_RZ,AGE_RZ,,Continuous,tutorial-biolincc_camp,[],5.0,13.0,True,None,biolincc_camp
2,\tutorial-biolincc_camp\ANYPET\,ANYPET,ANYPET,,Categorical,tutorial-biolincc_camp,"[No, Yes]",NaN,NaN,True,None,biolincc_camp
3,\tutorial-biolincc_camp\ANY_SMOKES\,ANY_SMOKES,ANY_SMOKES,,Categorical,tutorial-biolincc_camp,"[No, Yes]",NaN,NaN,True,None,biolincc_camp
4,\tutorial-biolincc_camp\DEHUMID\,DEHUMID,DEHUMID,,Categorical,tutorial-biolincc_camp,"[DK, No, Yes]",NaN,NaN,True,None,biolincc_camp


PIC-SURE integrates clinical and genomic datasets across *BDC*, including TOPMed and TOPMed-related studies, COVID-19 studies, and BioLINCC studies. Each variable is organized as a concept path that contains information about the study, variable group, and variable. Though the specifics of the concept paths are dependent on the type of study, the overall information included is the same. 

Data Organization in PIC-SURE
---------------------------------------
| Data organization | TOPMed & TOPMed-related studies | BioLINCC & COVID-19 studies (including public data) |
|-------------------|---------------------------------|-----------------------------|
| General organization | Data organized using the format implemented by the database of Genotypes and Phenotypes (dbGaP). Generally, a given study will have several tables, and those tables have several variables. | Data do not follow dbGaP format; there are no phv or pht accessions. Data are organized in groups of like variables, when available. For example, variables like Age, Gender, and Race could be part of the Demographics variable group. |
| Concept path structure | \phs\pht\phv\variable name\ | \phs\variable name |
| Variable ID | phv corresponding to the variable accession number | Equivalent to variable name | 
| Variable name | Encoded variable name that was used by the original submitters of the data | Encoded variable name that was used by the original submitters of the data |
| Variable description | Description of the variable | Description of the variable, as available |
| Dataset ID | pht corresponding to the trait table accession number | Equivalent to Dataset name | 
| Dataset name | Name of the trait table | Name of a group of like variables, as available | 
| Dataset description | Description of the trait table | Description of a group of like variables, as available |
| Study ID | phs corresponding to the study accession number | phs corresponding to the study accession number |
| Study description | Description of the study from dbGaP | Description of the study from dbGaP |

*Note: The concept paths in PIC-SURE are used for querying. These are stored in the `conceptPath` column of the data dictionary shown above.*

The `conceptPath` column contains the concept paths for specific variables, which are used for building cohorts and selecting participant-level data.

In [10]:
my_variables_df["conceptPath"].head(10).tolist()  # What are the first ten concept paths in my search results?

['\\tutorial-biolincc_camp\\AGEHOME\\',
 '\\tutorial-biolincc_camp\\AGE_RZ\\',
 '\\tutorial-biolincc_camp\\ANYPET\\',
 '\\tutorial-biolincc_camp\\ANY_SMOKES\\',
 '\\tutorial-biolincc_camp\\DEHUMID\\',
 '\\tutorial-biolincc_camp\\ETHNIC\\',
 '\\tutorial-biolincc_camp\\FDAYS\\',
 '\\tutorial-biolincc_camp\\GENDER\\',
 '\\tutorial-biolincc_camp\\HEMOG\\',
 '\\tutorial-biolincc_camp\\ID\\']

We can also view additional information for individual variables by filtering the DataFrame on `conceptPath`.

In [11]:
first_var = my_variables_df["conceptPath"].iloc[0]  # Save the concept path for the first variable
my_variables_df[my_variables_df["conceptPath"] == first_var]  # Show information for that first concept path

,conceptPath,name,display,description,dataType,studyId,values,min,max,allowFiltering,meta,studyAcronym
0,\tutorial-biolincc_camp\AGEHOME\,AGEHOME,AGEHOME,,Categorical,tutorial-biolincc_camp,"[0, 1, 10, 100, 102, 103, 104, 105, 11, 110, 1...",NaN,NaN,True,None,biolincc_camp


Now you can try to search for a term on your own. Below is sample code on how to search for the term `sex`. To practice searching the data dictionary, you can change "sex" to a term you are interested in. You will see the results displayed as a DataFrame. Note - the results displayed will show results from all studies you have access to.

In [12]:
my_search = session.searchDictionary("sex")  # Change "sex" to a term you are interested in
#my_search.head()  # Show the variables that match my search result

## Using PIC-SURE to build a query and retrieve data
You can also use the PIC-SURE API to build a query and retrieve data. With this functionality, you can filter based on specific variables, add others, and export the data as a DataFrame into this notebook.

Queries are built by assembling clauses and clause groups, then passing them to `picsure.buildQuery()` and running with `session.runQuery()`.

There are several clause types that can be used to build a query, which are listed below.

| Clause type (`PhenotypicFilterType`) | Arguments | Effect |
|--------------------------------------|-----------|--------|
| `FILTER` | concept path(s), `categories=` or `min=`/`max=` | Keeps only records matching the criteria; add the path to `includeConcepts` to also return it as a column |
| `REQUIRE` | concept path(s) | Only records that have non-null values for **all** listed variables |
| `ANYRECORD` | concept path(s) | Only records that have at least one non-null value for the listed variables |

Use `picsure.buildClause(keys, type, ...)` to create a single clause, `picsure.buildClauseGroup([...], operator=picsure.GroupOperator.AND)` to combine clauses, and `picsure.buildQuery(phenotypicFilter=..., includeConcepts=[...])` to assemble the final query. `includeConcepts` adds output columns without additional filtering (equivalent to the former `select()`).

As an example query, let's use the Framingham tutorial dataset to investigate the prevalence of hypertension and distribution of age of current smokers with body mass index greater than 20.

In [13]:
# Ensure that only Framingham tutorial variables are shown in the data dictionary, which can vary based on individual access
phs_number = "tutorial-biolincc_framingham"
tutorial_df = my_variables_df[my_variables_df.studyId == phs_number]

### Build a query with a categorical variable - Current smoker
Let's practice building a query by filtering on variables. Based on the search for the Framingham tutorial dataset variables, we can save the concept path of the "Current cigarette smoking at exam" variable, which is a categorical variable. 

In [14]:
smoke_variable_path = tutorial_df.conceptPath[tutorial_df.description == "Current cigarette smoking at exam"].item()
smoke_variable_path

'\\tutorial-biolincc_framingham\\CURSMOKE\\'

We can take a look at the options of values to filter by using the `values` column of the data dictionary.

In [15]:
tutorial_df["values"][tutorial_df.description == "Current cigarette smoking at exam"]

101    [Current smoker, Not current smoker]
Name: values, dtype: object

Let's apply a filter on the "Current cigarette smoking at exam" variable to only select participants with "Current smoker." Note that though we are only filtering by one value, you can filter by multiple values by passing a list to the `categories=` argument of `picsure.buildClause(...)`.

In [16]:
smoke_clause = picsure.buildClause(smoke_variable_path, picsure.PhenotypicFilterType.FILTER, categories="Current smoker")

### Build a query with a continuous variable - BMI

Let's practice building a query by filtering on a continuous variable, in this case, BMI. We can find the BMI concept path using a similar approach as above.

In [17]:
bmi_variable_path = tutorial_df.conceptPath[tutorial_df.name == "BMI"].item()
bmi_variable_path

'\\tutorial-biolincc_framingham\\BMI\\'

We can look at the minimum and maximum values of the variable using the `min` and `max` columns of the data dictionary.

In [18]:
tutorial_df[["min", "max"]][tutorial_df.name == "BMI"]

,min,max
98,14.43,56.8


Let's apply a filter on the "Body Mass Index, weight in kilograms/height meters squared" variable to select only participants with values greater than 20. Note that while in this example only a `min` is specified, a `max` can also be defined for the filter.

In [19]:
bmi_clause = picsure.buildClause(bmi_variable_path, picsure.PhenotypicFilterType.FILTER, min=20)

### Adding variables to include in export - Age and Hypertension
In addition to adding filters, specific variables can be included in the export for analysis. Let's do this for the "Age at exam (years)" and "Hypertensive. Defined as the first exam treated for high blood pressure or second exam in which either Systolic is 6 140 mmHg or Diastolic 6 90mmHg" variables.

In [20]:
age_variable_path = tutorial_df.loc[tutorial_df.description == "Age at exam (years)", "conceptPath"].item()
hyperten_variable_path = tutorial_df.loc[tutorial_df.name == "HYPERTEN", "conceptPath"].item()

Let's add these variables to our query using a `REQUIRE` clause, which will only select participants that have non-null values for both of these variables. We will also pass all four concept paths to `includeConcepts` in `buildQuery()` so they appear as output columns.

In [27]:
# REQUIRE applies per concept path; a single multi-path clause would OR them.
# To require BOTH age and hypertension, build one REQUIRE clause per path and AND them.
age_require_clause = picsure.buildClause(age_variable_path, picsure.PhenotypicFilterType.REQUIRE)
hyperten_require_clause = picsure.buildClause(hyperten_variable_path, picsure.PhenotypicFilterType.REQUIRE)

filter_tree = picsure.buildClauseGroup([smoke_clause, bmi_clause, age_require_clause, hyperten_require_clause], operator=picsure.GroupOperator.AND)
example_query = picsure.buildQuery(
    phenotypicFilter=filter_tree,
    includeConcepts=["\\tutorial-biolincc_framingham\\STROKE\\"],
)

### Running the query and exporting participant-level data
The query has been constructed and can now be run. `session.runQuery()` with `type=picsure.QueryType.PARTICIPANT` returns a pandas DataFrame of participant-level data.

In the data dictionary dataframe shown previously, each row represented a single concept path or variable. In the query dataframe, the concept paths are added as columns with each row representing a participant with data that matches your query. 

The dataframe above should contain some automatically exported concept paths, such as `patient_id`, `Parent Study Accession with Subject ID`, `Topmed Study Accession with Subject ID`, and `consents`, and the concept paths we added to our query.

In [28]:
example_results = session.runQuery(example_query, type=picsure.QueryType.PARTICIPANT)
example_results.head()

,patient_id,\tutorial-biolincc_framingham\STROKE\,\tutorial-biolincc_framingham\CURSMOKE\,\tutorial-biolincc_framingham\BMI\,\tutorial-biolincc_framingham\AGE\,\tutorial-biolincc_framingham\HYPERTEN\
0,192514,Event did not occur during followup,Current smoker\tNot current smoker,20.52\t21.18\t22.17,64.0\t70.0\t76.0,Event did occur during followup
1,192513,Event did not occur during followup,Current smoker,20.83\t20.88,44.0\t50.0,Event did not occur during followup
2,192523,Event did not occur during followup,Current smoker,22.5\t22.51,46.0\t52.0\t58.0,Event did occur during followup
3,192526,Event did not occur during followup,Current smoker\tNot current smoker,25.16\t27.42\t27.82,35.0\t41.0\t47.0,Event did occur during followup
4,192524,Event did not occur during followup,Current smoker,34.34\t34.69\t36.81,53.0\t59.0\t65.0,Event did occur during followup


As you can see in the results, there are some instances where study participants may have more than one value for a given variable. For example, this may be the case when a study participants answers questionnaires for multiple visits. 

In the PIC-SURE output, this is shown as values being separated by a tab or `\t` value. These multiple values will need to be accounted for depending on the planned analysis.

With this example, averages of the age and BMI values could be calculated and a new variable "ever smoker" could be created based on whether "current smoker" was ever answered for the CURSMOKE variable. The code below shows this example of how to handle these values.

*Note: Approaches to handling multiple values will differ based on the approach.*

In [23]:
# Select rows of interest and rename them
clean_results = example_results[["\\tutorial-biolincc_framingham\\AGE\\", "\\tutorial-biolincc_framingham\\BMI\\", "\\tutorial-biolincc_framingham\\CURSMOKE\\", "\\tutorial-biolincc_framingham\\HYPERTEN\\"]]
clean_results.columns = ["AGE", "BMI", "CURSMOKE", "HYPERTEN"]

In [24]:
# Function that splits the values and calculates the mean
import statistics
def mean_multiple_values(df_values):
    sep_values = str(df_values).split("\t")
    mean_val = statistics.mean([float(i) for i in sep_values])
    return(mean_val)

# Apply the function to calculate means to the AGE and BMI variables
clean_results.loc[:, "mean_age"] = clean_results.loc[:, "AGE"].apply(mean_multiple_values)
clean_results.loc[:, "mean_bmi"] = clean_results.loc[:, "BMI"].apply(mean_multiple_values)

In [25]:
# Function that flags participants as smoker if they have an answer of "Current smoker"
def ever_smoker(smoke_vals):
    sep_smoke_vals = smoke_vals.split("\t")
    if "Current smoker" in sep_smoke_vals:
        return("Smoker")
    else:
        return("Non-smoker")
    
# Apply the function to identify smokers to the CURSMOKE variable
clean_results.loc[:, "ever_smoker"] = clean_results.loc[:, "CURSMOKE"].apply(ever_smoker)

In [26]:
# Take a look at the new results
clean_results[["mean_age", "mean_bmi", "ever_smoker","HYPERTEN"]]

,mean_age,mean_bmi,ever_smoker,HYPERTEN
0,70.000000,21.290000,Smoker,Event did occur during followup
1,47.000000,20.855000,Smoker,Event did not occur during followup
2,52.000000,22.505000,Smoker,Event did occur during followup
3,41.000000,26.800000,Smoker,Event did occur during followup
4,59.000000,35.280000,Smoker,Event did occur during followup
...,...,...,...,...
2242,39.000000,29.700000,Smoker,Event did occur during followup
2243,57.000000,23.870000,Smoker,Event did occur during followup
2244,65.000000,27.343333,Smoker,Event did occur during followup
2245,53.666667,27.616667,Smoker,Event did occur during followup
